# Занятие 7, пара 1. SQL для аналитики. CTE и оконные функции


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Когда одной строки на группу уже мало

В прошлой паре GROUP BY помогал свернуть события в итог. Сегодня хочется оставить каждую
строку и рядом подписать: какое у неё место, сколько накопилось к этому моменту, что было перед ней.
Это работа **оконных функций**. Слово «окно» здесь означает набор строк для вычисления,
а не окошко приложения. **CTE** поможет разложить длинный запрос на понятные шаги.

Нужны SELECT, WHERE, GROUP BY и ORDER BY из 06.2. Синтаксис новых конструкций разбираем до применения.
SQLite и Pandas достаточны, интернет не нужен. Все таблицы загрузим заново.
К концу пары вы сможете разбить запрос на этапы, выбрать правило при равных результатах,
найти первое достижение порога и отличить предыдущую запись от предыдущего календарного дня.
Результат — три обоснованных запроса; каждое решение придётся сверить со смыслом вопроса.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Прочитаем десять строк

Знакомый json.load читает словарь из файла; DataFrame превращает список записей в таблицу.
`day` — номер условного дня. Если дня нет, мы не знаем его значение. `revenue` измеряется
в условных денежных единицах; это вымышленные показатели, не данные магазина.


In [ ]:
import json
import sqlite3
import pandas as pd
from IPython.display import display

with (DATA_DIR / 'warehouse' / 'demo.json').open(encoding='utf-8') as file:
    source = json.load(file)
daily = pd.DataFrame(source['daily'])
connection = sqlite3.connect(':memory:')
daily.to_sql('daily', connection, index=False, if_exists='replace')
print('SQLite:', sqlite3.sqlite_version)
display(daily)


Давайте вспомним обычную группировку. У каждого магазина получим одну строку.
`MAX` возвращает наибольшее известное значение, `MIN` — наименьшее, `AVG` — среднее.
Сама дата максимума в результате не появится: мы её не просили и не объяснили, какую выбрать при равенстве.


In [ ]:
query = """
SELECT store, AVG(revenue) AS mean_revenue, MAX(revenue) AS best_revenue
FROM daily GROUP BY store ORDER BY store
"""
display(pd.read_sql_query(query, connection))


### 2. Дадим промежуточному результату имя

`WITH totals AS (...)` — именованный этап запроса: CTE, Common Table Expression.
В скобках обычный SELECT. После закрывающей скобки основной запрос может читать `totals`
как таблицу. Это имя действует только в одном SQL-запросе; новый файл и постоянная таблица не создаются.
Пишем сначала понятный внутренний запрос, затем внешний фильтр.

Сначала прочитайте внутренний SELECT отдельно: он возвращает две суммы. Затем во внешнем SELECT остаётся один вопрос — какие из этих сумм не меньше 400? Не нужно одновременно держать в голове всю исходную таблицу.


In [ ]:
query = """
WITH totals AS (
    SELECT store, SUM(revenue) AS total
    FROM daily GROUP BY store
)
SELECT store, total
FROM totals
WHERE total >= 400
ORDER BY store
"""
display(pd.read_sql_query(query, connection))


Здесь можно было обойтись HAVING. CTE полезен не как обязательная обёртка, а когда шаги
нужно разделить. Можно объявить несколько этапов через запятую: следующий использует предыдущий.
[WITH в SQLite](https://sqlite.org/lang_with.html).

### 3. Среднее по группе, но строки сохраняются

Добавим `OVER (PARTITION BY store)` после AVG. PARTITION BY разбивает строки по магазинам,
а окно вычисляет среднее для каждого такого набора. Каждая исходная строка остаётся на месте,
только рядом появляется общий показатель. Это отличие от GROUP BY и есть главный смысл опыта.


In [ ]:
query = """
SELECT store, day, revenue,
       AVG(revenue) OVER (PARTITION BY store) AS store_mean
FROM daily
ORDER BY store, day
"""
display(pd.read_sql_query(query, connection))


Хотим оставить дни выше среднего магазина. Оконное значение нельзя отфильтровать через
WHERE на том же уровне SELECT: это другой этап обработки. Сначала посчитаем его в CTE,
затем внешний запрос сравнит две обычные колонки. Для MAX/SUM правило OVER такое же.


In [ ]:
query = """
WITH marked AS (
    SELECT store, day, revenue,
           AVG(revenue) OVER (PARTITION BY store) AS store_mean
    FROM daily
)
SELECT store, day, revenue, store_mean
FROM marked WHERE revenue > store_mean
ORDER BY store, day
"""
display(pd.read_sql_query(query, connection))


### 4. Какое место заняла строка?

`ROW_NUMBER()` нумерует строки от 1 внутри группы. Порядок **внутри OVER** определяет,
кто получает номер; внешний ORDER BY определяет, как мы показываем результат.
Для равных сумм добавим day: более ранний день будет первым. Это явное правило, а не случайность движка.


In [ ]:
query = """
SELECT store, day, revenue,
       ROW_NUMBER() OVER (PARTITION BY store ORDER BY revenue DESC, day) AS position
FROM daily ORDER BY store, position
"""
display(pd.read_sql_query(query, connection))


А если равные суммы должны делить место? `RANK()` даёт им одинаковый ранг,
после равенства оставляет пропуск:1,2,2,4. Внутри окна сравниваем только revenue.
Если добавить day туда же, дни перестанут быть равными по условию ранжирования.
Внешняя сортировка по day может упорядочить показ, не меняя смысл ранга.


In [ ]:
query = """
SELECT store, day, revenue,
       RANK() OVER (PARTITION BY store ORDER BY revenue DESC) AS place
FROM daily ORDER BY store, place, day
"""
display(pd.read_sql_query(query, connection))


### ✏️ Ваш ход — проверить условие заказчика

Заказчик попросил «показать все лучшие дни каждого магазина».
Коллега предлагает `ORDER BY revenue DESC LIMIT 2`; другой — взять ROW_NUMBER=1 в каждой группе.
Объясните, почему оба предложения могут потерять нужные строки на наших данных.
Напишите собственный `my_best` со столбцами store,day,revenue, отсортированный по store,day.
Можно выбрать RANK или MAX с окном — обоснуйте свой выбор. Должны сохраняться все равенства на первом месте.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.1.1', my_best)


**Мой вывод:** …


### 5. Сколько накопилось к текущему дню?

Понадобится SUM с порядком по day. Явно зададим рамку:
`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — от первой строки группы до текущей включительно.
SUM работает именно по этой рамке, не по всей группе сразу. Пара(store,day) у нас уникальна.
Если в день несколько событий, сначала решают: нужен итог дня или последовательность событий с дополнительным ключом.


In [ ]:
query = """
SELECT store, day, revenue,
       SUM(revenue) OVER (
           PARTITION BY store ORDER BY day
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM daily ORDER BY store, day
"""
display(pd.read_sql_query(query, connection))


Не будем полагаться на неявную рамку: равные значения ключа сортировки могут изменить
ожидаемый результат. Проверьте первые три строки Севера вручную:100,200,260.
Мы суммируем только предоставленные записи. Отсутствующий третий день не превращается в доказанный ноль.


### ✏️ Ваш ход — найти первый момент, а не весь хвост

Для каждого магазина найдите первый записанный день,
к которому накопленная сумма достигла 150. Результат `my_first`: store,first_day по store.
Не возвращайте все последующие дни. Разбейте решение на расчёт накопления, фильтр и выбор первого дня.
В выводе уточните: что мы можем утверждать о пропущенных днях и чего не можем?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.1.2', my_first)


**Мой вывод:** …


### 6. Предыдущая запись — не обязательно вчера

`LAG(revenue)` берёт значение из предыдущей строки в заданном порядке, первая строка получает NULL.
`LEAD` аналогично смотрит вперёд; сегодня достаточно LAG. У каждого магазина своё окно,
иначе последний день одного магазина можно случайно сравнить с первым днём другого.


In [ ]:
query = """
SELECT store, day, revenue,
       LAG(revenue) OVER (PARTITION BY store ORDER BY day) AS previous_revenue,
       LAG(day) OVER (PARTITION BY store ORDER BY day) AS previous_day
FROM daily ORDER BY store, day
"""
display(pd.read_sql_query(query, connection))


У Севера перед днём 4 идёт день 2. Разность выручки между этими строками существует,
но назвать её «изменением за один день» нельзя. Вынесем значения в CTE и вычислим разности
во внешнем запросе. Не заменяем NULL предыдущего значения нулём: так появилось бы выдуманное изменение.


In [ ]:
query = """
WITH previous AS (
    SELECT store, day, revenue,
           LAG(revenue) OVER (PARTITION BY store ORDER BY day) AS previous_revenue,
           LAG(day) OVER (PARTITION BY store ORDER BY day) AS previous_day
    FROM daily
)
SELECT store, day, day-previous_day AS gap, revenue-previous_revenue AS change
FROM previous ORDER BY store, day
"""
display(pd.read_sql_query(query, connection))


### ✏️ Ваш ход — исправить ложную тревогу

Нужен список падений выручки хотя бы на 30 **между соседними днями**.
Коллега просто фильтрует change<=−30 и объявляет тревогу. На какой строке это создаёт
необоснованное сообщение про сутки? Напишите `my_alerts`: store,day,change по store,day.
Используйте и размер изменения, и расстояние между днями. Первая строка магазина не должна стать тревогой.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('07.1.3', my_alerts)


**Мой вывод:** …


### Итог и ещё один вопрос

CTE отделяет этапы. Окно сохраняет исходные строки. Ранжирование требует договора о равенствах,
накопление — порядка и рамки, сравнение с предыдущим — проверки интервала.
Напишите к своему запросу одно условие, без которого его вывод стал бы неверным.

**Резерв 10–15 минут:** один магазин мог прислать два обновления показателя за один день.
Предложите, какие дополнительные поля нужны, чтобы выбрать последнюю версию, а не максимальную сумму.
Создайте 3–4 строки такого примера и решите через ROW_NUMBER. Не удаляйте дубли из основной daily.

[Справочник окон SQLite](https://sqlite.org/windowfunctions.html) — найдите только ROW_NUMBER,
RANK, LAG и определение рамки. Понимать четыре примера полезнее, чем выписать весь список функций.
